In [2]:
import pandas as pd

df = pd.read_csv("data/faqs.csv")

# This styles the table so all text and headers align neatly to the left
df.head().style.set_properties(**{'text-align': 'left'}).set_table_styles([
    dict(selector='th', props=[('text-align', 'left')])
])


,id,question,answer
0,1,How do I create an account?,Select Sign Up and enter your email address.
1,2,How can I reset my password?,Select Forgot Password on the login page.
2,3,How do I enroll in a course?,Open the course page and select Enroll.
3,4,Can I study on my phone?,"Yes, use a compatible mobile browser."
4,5,How do I download my certificate?,Complete the course requirements and open the certificate section.


In [3]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df[["question", "answer"]].isna().sum())

print("\nDuplicate FAQ pairs:")
print(df.duplicated(subset=["question", "answer"]).sum())

Dataset shape: (10, 3)

Missing values:
question    0
answer      0
dtype: int64

Duplicate FAQ pairs:
0


In [4]:
# 1. Create a copy of the original dataset
df_clean = df.copy()

# 2. Remove leading and trailing spaces
for column in ["question", "answer"]:
    df_clean[column] = df_clean[column].str.strip()

# 3. Remove records with empty questions or answers
df_clean = df_clean[
    (df_clean["question"] != "")
    & (df_clean["answer"] != "")
]

# 4. Remove duplicate FAQ pairs
df_clean = df_clean.drop_duplicates(
    subset=["question", "answer"]
).reset_index(drop=True)

# 5. Verify the result
print("Original rows:", len(df))
print("Clean rows:", len(df_clean))

df_clean.head().style.set_properties(**{'text-align': 'left'}).set_table_styles([
    dict(selector='th', props=[('text-align', 'left')])
])


Original rows: 10
Clean rows: 10


,id,question,answer
0,1,How do I create an account?,Select Sign Up and enter your email address.
1,2,How can I reset my password?,Select Forgot Password on the login page.
2,3,How do I enroll in a course?,Open the course page and select Enroll.
3,4,Can I study on my phone?,"Yes, use a compatible mobile browser."
4,5,How do I download my certificate?,Complete the course requirements and open the certificate section.


In [5]:
# Create a separate DataFrame for our searchable text
df_chunks = df_clean.copy()

# Combine each question and answer into one text chunk
df_chunks["text"] = (
    "Question: " + df_chunks["question"]
    + "\nAnswer: " + df_chunks["answer"]
)

# Convert the text column into a Python list
chunks = df_chunks["text"].tolist()

# Inspect the results
print("Number of chunks:", len(chunks))
print("\nExample chunk:\n")
print(chunks[0])

Number of chunks: 10

Example chunk:

Question: How do I create an account?
Answer: Select Sign Up and enter your email address.


In [6]:
import chromadb

# Create a persistent ChromaDB database
client = chromadb.PersistentClient(path="chroma_db")

# Create or access our FAQ collection
collection = client.get_or_create_collection(
    name="faq_collection"
)

print("ChromaDB collection is ready.")

ChromaDB collection is ready.


In [12]:
# Create a unique ID for each FAQ
ids = [f"faq-{faq_id}" for faq_id in df_chunks["id"]]

# Store the question separately as searchable metadata
metadatas = [
    {"faq_id": str(faq_id), "question": question}
    for faq_id, question in zip(
        df_chunks["id"],
        df_chunks["question"]
    )
]

# Store the documents and generate their embeddings
collection.upsert(
    ids=ids,
    documents=chunks,
    metadatas=metadatas
)

print("Records stored:", collection.count())

Records stored: 10


In [13]:
query = "I forgot my password. How can I get back into my account?"

results = collection.query(
    query_texts=[query],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for rank, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\nMatch {rank}")
    print("Question:", metadata["question"])
    print("Document:", document)
    print("Distance:", round(distance, 4))


Match 1
Question: How can I reset my password?
Document: Question: How can I reset my password?
Answer: Select Forgot Password on the login page.
Distance: 0.4911

Match 2
Question: What should I do if I cannot log in?
Document: Question: What should I do if I cannot log in?
Answer: Check your email and password, or reset your password.
Distance: 0.7177

Match 3
Question: How can I contact support?
Document: Question: How can I contact support?
Answer: Open the Help Center and submit a support request.
Distance: 1.2939
